# Методы обработки языковых последовательностей: классификация, разметка, суммаризация

Ноутбук к отчёту по проектно-технологической практике. Три задачи, для каждой из которых выстроена последовательность методов (от простого к сложному):

1. SST-2: классификация предложения (TF-IDF + LR, BiLSTM, DistilBERT, заморозка слоёв, LoRA, ансамбль).
2. CoNLL-2003: NER (CRF, BiLSTM, DistilBERT, взвешенная функция потерь, раздельные скорости обучения).
3. CNN/DailyMail: суммаризация (lead-3, T5-small, дообучение, LoRA, параметры декодирования).

Запуск: Runtime → Change runtime type → T4 GPU, затем Run all. Параметр `PROFILE` ниже: `smoke` для проверки (несколько минут), `standard` для цифр в отчёте.

In [20]:
!pip -q uninstall -y torchao
!pip -q install rouge-score sklearn-crfsuite peft

In [21]:
PROFILE = "standard"   # "smoke" или "standard"

import os, json, time, re, math, itertools, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = "results"
os.makedirs(f"{OUT}/figures", exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

CFG = {
    "standard": dict(sst_train=10000, sst_seeds=[0, 1, 2], sst_epochs=2,
                     ner_train=6000, ner_seeds=[0, 1], ner_epochs=3, lstm_epochs=12,
                     sum_train=2000, sum_test=300, sum_epochs=2, sum_seeds=[0], n_boot=1000),
    "smoke":    dict(sst_train=1000, sst_seeds=[0], sst_epochs=1,
                     ner_train=600, ner_seeds=[0], ner_epochs=1, lstm_epochs=3,
                     sum_train=64, sum_test=16, sum_epochs=1, sum_seeds=[0], n_boot=100),
}[PROFILE]
json.dump(dict(profile=PROFILE, **CFG), open(f"{OUT}/run_config.json", "w"), indent=2)
print(DEVICE, torch.__version__)

cuda 2.14.1+cu130


In [22]:
import random
import re
import numpy as np


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def bootstrap_stat(arrays, stat_fn, n_boot=1000, alpha=0.05, seed=0):
    # arrays: по одному массиву на запуск (seed); объекты теста пересэмплируются
    # одним набором индексов для всех запусков, статистика усредняется по запускам
    rng = np.random.default_rng(seed)
    n = len(arrays[0])
    point = float(np.mean([stat_fn(a) for a in arrays]))
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boots[b] = np.mean([stat_fn(a[idx]) for a in arrays])
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return point, float(lo), float(hi)


def paired_bootstrap(arrays_a, arrays_b, stat_fn, n_boot=1000, alpha=0.05, seed=0):
    # разность статистик A - B на одних и тех же пересэмплированных объектах
    rng = np.random.default_rng(seed)
    n = len(arrays_a[0])
    diff = float(np.mean([stat_fn(a) for a in arrays_a]) - np.mean([stat_fn(b) for b in arrays_b]))
    boots = np.empty(n_boot)
    for k in range(n_boot):
        idx = rng.integers(0, n, n)
        sa = np.mean([stat_fn(a[idx]) for a in arrays_a])
        sb = np.mean([stat_fn(b[idx]) for b in arrays_b])
        boots[k] = sa - sb
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return diff, float(lo), float(hi), float((boots <= 0).mean())


def acc_stat(a):
    return float((a[:, 0] == a[:, 1]).mean())


def macro_f1_stat(a):
    from sklearn.metrics import f1_score
    return float(f1_score(a[:, 0], a[:, 1], average="macro"))


def get_entities(tags):
    entities = []
    start, kind = None, None
    for i, tag in enumerate(list(tags) + ["O"]):
        if tag.startswith("B-") or tag == "O" or (tag.startswith("I-") and tag[2:] != kind):
            if kind is not None:
                entities.append((kind, start, i - 1))
                start, kind = None, None
            if tag.startswith("B-") or tag.startswith("I-"):
                start, kind = i, tag[2:]
    return set(entities)


def entity_counts(gold_seqs, pred_seqs):
    counts = []
    for gold, pred in zip(gold_seqs, pred_seqs):
        g = get_entities(gold)
        p = get_entities(pred)
        counts.append((len(g & p), len(p), len(g)))
    return np.array(counts, dtype=float)


def entity_f1_stat(counts):
    tp, n_pred, n_gold = counts.sum(axis=0)
    prec = tp / n_pred if n_pred else 0.0
    rec = tp / n_gold if n_gold else 0.0
    return float(2 * prec * rec / (prec + rec)) if prec + rec else 0.0


def rouge_per_example(preds, refs):
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rows = []
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        rows.append([s["rouge1"].fmeasure, s["rouge2"].fmeasure, s["rougeL"].fmeasure])
    return np.array(rows)


def lead3(article):
    sentences = re.split(r"(?<=[.!?])\s+", article.strip())
    return " ".join(sentences[:3])

In [23]:
ROWS = []      # сводная таблица результатов
HIST = {}      # кривые обучения
PRED = {}      # массивы для бутстрепа и графиков

def add_row(task, method, metrics, extra=None):
    row = dict(task=task, method=method, **metrics)
    if extra:
        row.update(extra)
    ROWS.append(row)
    pd.DataFrame(ROWS).to_csv(f"{OUT}/results.csv", index=False)

def fmt_ci(p, lo, hi, scale=100):
    return f"{p*scale:.1f} [{lo*scale:.1f}; {hi*scale:.1f}]"

def collate_pad(tok, feats, extra_labels=None, pad_label=-100):
    batch = tok.pad(feats, return_tensors="pt")
    if extra_labels is not None:
        width = batch["input_ids"].shape[1]
        batch["labels"] = torch.tensor([l + [pad_label] * (width - len(l)) for l in extra_labels])
    return batch

def to_dev(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

def make_scheduler(opt, total, warmup=0.1):
    from transformers import get_linear_schedule_with_warmup
    return get_linear_schedule_with_warmup(opt, int(total * warmup), total)

## Задача 1. SST-2

In [24]:
from datasets import load_dataset
sst = load_dataset("nyu-mll/glue", "sst2")
train_all = sst["train"].shuffle(seed=42).select(range(CFG["sst_train"]))
sst_tr_x, sst_tr_y = list(train_all["sentence"]), list(train_all["label"])
sst_te_x, sst_te_y = list(sst["validation"]["sentence"]), np.array(sst["validation"]["label"])
print(len(sst_tr_x), len(sst_te_x), np.bincount(sst_te_y))

10000 872 [428 444]


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(sst_tr_x)
lr = LogisticRegression(max_iter=1000, C=4.0).fit(Xtr, sst_tr_y)
p_lr = lr.predict_proba(vec.transform(sst_te_x))[:, 1]
PRED["sst_tfidf"] = [np.c_[sst_te_y, (p_lr > 0.5).astype(int)]]
PRED["sst_tfidf_prob"] = p_lr
print("TF-IDF + LR accuracy:", acc_stat(PRED["sst_tfidf"][0]))

TF-IDF + LR accuracy: 0.7878440366972477


In [26]:
def tok_simple(s):
    return re.findall(r"[a-z0-9']+|[^\sa-z0-9]", s.lower())

class LstmCls(nn.Module):
    def __init__(self, vocab, dim=128, hid=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.rnn = nn.LSTM(dim, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.out = nn.Linear(2 * hid, 2)
    def forward(self, ids, lens):
        x = self.drop(self.emb(ids))
        packed = nn.utils.rnn.pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.rnn(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True)
        mask = (torch.arange(h.shape[1], device=h.device)[None, :] < lens[:, None]).unsqueeze(-1)
        pooled = h.masked_fill(~mask, -1e9).max(dim=1).values
        return self.out(self.drop(pooled))

def run_lstm_cls(seed):
    set_seed(seed)
    freq = {}
    for s in sst_tr_x:
        for w in tok_simple(s):
            freq[w] = freq.get(w, 0) + 1
    vocab = {w: i + 2 for i, w in enumerate(w for w, c in freq.items() if c >= 2)}
    enc = lambda s: [vocab.get(w, 1) for w in tok_simple(s)] or [1]
    tr = [enc(s) for s in sst_tr_x]
    te = [enc(s) for s in sst_te_x]
    def batchify(seqs):
        lens = torch.tensor([len(s) for s in seqs])
        ids = torch.zeros(len(seqs), int(lens.max()), dtype=torch.long)
        for i, s in enumerate(seqs):
            ids[i, :len(s)] = torch.tensor(s)
        return ids.to(DEVICE), lens.to(DEVICE)
    model = LstmCls(len(vocab) + 2).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    y = torch.tensor(sst_tr_y)
    losses = []
    for ep in range(5):
        model.train()
        order = np.random.permutation(len(tr))
        tot = 0
        for i in range(0, len(order), 64):
            idx = order[i:i + 64]
            ids, lens = batchify([tr[j] for j in idx])
            loss = nn.functional.cross_entropy(model(ids, lens), y[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        losses.append(tot / len(order))
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(te), 256):
            ids, lens = batchify(te[i:i + 256])
            probs.append(torch.softmax(model(ids, lens), -1)[:, 1].cpu().numpy())
    return np.concatenate(probs), losses

arrs = []
for sd in CFG["sst_seeds"]:
    p, l = run_lstm_cls(sd)
    arrs.append(np.c_[sst_te_y, (p > 0.5).astype(int)])
HIST["sst_bilstm"] = l
PRED["sst_bilstm"] = arrs
print("BiLSTM accuracy:", np.mean([acc_stat(a) for a in arrs]))

BiLSTM accuracy: 0.7748470948012232


In [27]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

SST_MODEL = "distilbert-base-uncased"
sst_tok = AutoTokenizer.from_pretrained(SST_MODEL)

def encode_cls(texts):
    e = sst_tok(texts, truncation=True, max_length=64)
    return [{k: e[k][i] for k in e} for i in range(len(texts))]

sst_tr_enc, sst_te_enc = encode_cls(sst_tr_x), encode_cls(sst_te_x)

def run_distil(variant, seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(SST_MODEL, num_labels=2)
    lr_ = 5e-5
    if variant == "freeze":
        for p in model.distilbert.embeddings.parameters():
            p.requires_grad = False
        for layer in model.distilbert.transformer.layer[:4]:
            for p in layer.parameters():
                p.requires_grad = False
    elif variant == "lora":
        cfg = LoraConfig(task_type="SEQ_CLS", r=8, lora_alpha=16, lora_dropout=0.1,
                         target_modules=["q_lin", "v_lin"], modules_to_save=["pre_classifier", "classifier"])
        model = get_peft_model(model, cfg)
        lr_ = 5e-4
    model.to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr_, weight_decay=0.01)
    bs, epochs = 32, CFG["sst_epochs"]
    steps = epochs * math.ceil(len(sst_tr_enc) / bs)
    sched = make_scheduler(opt, steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    labels = torch.tensor(sst_tr_y)
    losses = []
    for ep in range(epochs):
        model.train()
        order = np.random.permutation(len(sst_tr_enc))
        tot = 0
        for i in range(0, len(order), bs):
            idx = order[i:i + bs]
            batch = to_dev(collate_pad(sst_tok, [sst_tr_enc[j] for j in idx]))
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(**batch).logits
            loss = nn.functional.cross_entropy(logits.float(), labels[idx].to(DEVICE))
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(idx)
        losses.append(tot / len(order))
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(sst_te_enc), 128):
            batch = to_dev(collate_pad(sst_tok, sst_te_enc[i:i + 128]))
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(**batch).logits
            probs.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    n_train = sum(p.numel() for p in params)
    del model, opt
    torch.cuda.empty_cache()
    return np.concatenate(probs), losses, n_train

SST_PROBS = {}
for variant in ["full", "freeze", "lora"]:
    t0 = time.time()
    arrs, probs = [], []
    for sd in CFG["sst_seeds"]:
        p, l, ntr = run_distil(variant, sd)
        arrs.append(np.c_[sst_te_y, (p > 0.5).astype(int)])
        probs.append(p)
    PRED[f"sst_{variant}"] = arrs
    SST_PROBS[variant] = probs
    HIST[f"sst_{variant}"] = l
    PRED[f"sst_{variant}_ntrain"] = ntr
    print(variant, "acc", round(np.mean([acc_stat(a) for a in arrs]), 4), "trainable", ntr, f"{time.time()-t0:.0f}s")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (11 outputs left out)

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lora acc 0.8826 trainable 739586 62s


In [28]:
# ансамбль: среднее вероятностей TF-IDF + LR и полного DistilBERT
arrs = []
for p in SST_PROBS["full"]:
    ens = (p + PRED["sst_tfidf_prob"]) / 2
    arrs.append(np.c_[sst_te_y, (ens > 0.5).astype(int)])
PRED["sst_ensemble"] = arrs

sst_methods = [("TF-IDF + LR", "sst_tfidf"), ("BiLSTM", "sst_bilstm"),
               ("DistilBERT (полное дообучение)", "sst_full"),
               ("DistilBERT (заморозка 4 слоёв)", "sst_freeze"),
               ("DistilBERT + LoRA", "sst_lora"),
               ("Ансамбль LR + DistilBERT", "sst_ensemble")]
for name, key in sst_methods:
    acc = bootstrap_stat(PRED[key], acc_stat, CFG["n_boot"])
    f1 = float(np.mean([macro_f1_stat(a) for a in PRED[key]]))
    sd = float(np.std([acc_stat(a) for a in PRED[key]])) if len(PRED[key]) > 1 else 0.0
    add_row("SST-2", name, dict(acc=acc[0], acc_lo=acc[1], acc_hi=acc[2], f1=f1, acc_std=sd),
            dict(trainable=PRED.get(key + "_ntrain")))
d = paired_bootstrap(PRED["sst_ensemble"], PRED["sst_full"], acc_stat, CFG["n_boot"])
print("ансамбль - полный DistilBERT:", d)
pd.DataFrame(ROWS)

ансамбль - полный DistilBERT: (0.006498470948012147, -0.0011467889908257645, 0.01414373088685017, 0.058)


,task,method,acc,acc_lo,acc_hi,f1,acc_std,trainable
0,SST-2,TF-IDF + LR,0.787844,0.760321,0.814220,0.787057,0.000000,NaN
1,SST-2,BiLSTM,0.774847,0.753058,0.795489,0.772654,0.008997,NaN
2,SST-2,DistilBERT (полное дообучение),0.886086,0.866590,0.904817,0.885919,0.002356,66955010.0
3,SST-2,DistilBERT (заморозка 4 слоёв),0.876147,0.854358,0.897181,0.875989,0.003376,14767874.0
4,SST-2,DistilBERT + LoRA,0.882645,0.863140,0.903679,0.882569,0.004425,739586.0
5,SST-2,Ансамбль LR + DistilBERT,0.892584,0.873834,0.910550,0.892409,0.003784,NaN


## Задача 2. CoNLL-2003 (NER)

In [29]:
conll = None
for name in ["lhoestq/conll2003", "lukasgarbas/conll-03", "eriktks/conll2003"]:
    try:
        conll = load_dataset(name)
        print("загружено:", name)
        break
    except Exception as e:
        print(name, "не загрузился:", str(e)[:100])
assert conll is not None, "не удалось загрузить CoNLL-2003"

DEFAULT_LABELS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
feat = conll["train"].features["ner_tags"]
LABELS = list(feat.feature.names) if hasattr(feat, "feature") and hasattr(feat.feature, "names") else DEFAULT_LABELS
to_str = lambda seq: [LABELS[t] if isinstance(t, int) else t for t in seq]
print(LABELS, {k: len(v) for k, v in conll.items()})

ner_train = conll["train"].shuffle(seed=42).select(range(CFG["ner_train"]))
ner_test = conll["test"]
tr_tokens, tr_tags = list(ner_train["tokens"]), [to_str(s) for s in ner_train["ner_tags"]]
te_tokens, te_tags = list(ner_test["tokens"]), [to_str(s) for s in ner_test["ner_tags"]]
print(len(tr_tokens), len(te_tokens))

загружено: lhoestq/conll2003
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'] {'train': 14041, 'validation': 3250, 'test': 3453}
6000 3453


In [30]:
import sklearn_crfsuite

def word_feats(sent, i):
    w = sent[i]
    f = {"bias": 1.0, "w.lower": w.lower(), "suf3": w[-3:], "suf2": w[-2:], "pre2": w[:2],
         "title": w.istitle(), "upper": w.isupper(), "digit": w.isdigit(), "hyphen": "-" in w}
    for off, tag in [(-1, "p"), (1, "n")]:
        j = i + off
        if 0 <= j < len(sent):
            f[tag + ".lower"] = sent[j].lower()
            f[tag + ".title"] = sent[j].istitle()
            f[tag + ".upper"] = sent[j].isupper()
        else:
            f["BOS" if off < 0 else "EOS"] = True
    return f

feats = lambda sents: [[word_feats(s, i) for i in range(len(s))] for s in sents]
crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
crf.fit(feats(tr_tokens), tr_tags)
crf_pred = crf.predict(feats(te_tokens))
PRED["ner_crf"] = [entity_counts(te_tags, crf_pred)]
print("CRF F1:", entity_f1_stat(PRED["ner_crf"][0]))

CRF F1: 0.7555027856425244


In [31]:
def casing(w):
    if w.isdigit(): return 1
    if w.islower(): return 2
    if w.isupper(): return 3
    if w[:1].isupper(): return 4
    return 5

class LstmTagger(nn.Module):
    def __init__(self, vocab, n_lab, dim=100, cdim=10, hid=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.cas = nn.Embedding(6, cdim, padding_idx=0)
        self.rnn = nn.LSTM(dim + cdim, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.4)
        self.out = nn.Linear(2 * hid, n_lab)
    def forward(self, ids, cas):
        x = self.drop(torch.cat([self.emb(ids), self.cas(cas)], -1))
        h, _ = self.rnn(x)
        return self.out(self.drop(h))

def run_lstm_tagger(seed):
    set_seed(seed)
    freq = {}
    for s in tr_tokens:
        for w in s:
            freq[w.lower()] = freq.get(w.lower(), 0) + 1
    vocab = {w: i + 2 for i, w in enumerate(w for w, c in freq.items() if c >= 2)}
    l2i = {l: i for i, l in enumerate(LABELS)}
    def enc(sent):
        return [vocab.get(w.lower(), 1) for w in sent], [casing(w) for w in sent]
    def batchify(sents, tags=None):
        L = max(len(s) for s in sents)
        ids = torch.zeros(len(sents), L, dtype=torch.long)
        cas = torch.zeros(len(sents), L, dtype=torch.long)
        lab = torch.full((len(sents), L), -100, dtype=torch.long)
        for i, s in enumerate(sents):
            a, b = enc(s)
            ids[i, :len(s)], cas[i, :len(s)] = torch.tensor(a), torch.tensor(b)
            if tags is not None:
                lab[i, :len(s)] = torch.tensor([l2i[t] for t in tags[i]])
        return ids.to(DEVICE), cas.to(DEVICE), lab.to(DEVICE)
    model = LstmTagger(len(vocab) + 2, len(LABELS)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    losses = []
    for ep in range(CFG["lstm_epochs"]):
        model.train()
        order = np.random.permutation(len(tr_tokens))
        tot = 0
        for i in range(0, len(order), 32):
            idx = order[i:i + 32]
            ids, cas, lab = batchify([tr_tokens[j] for j in idx], [tr_tags[j] for j in idx])
            loss = nn.functional.cross_entropy(model(ids, cas).transpose(1, 2), lab, ignore_index=-100)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        losses.append(tot / len(order))
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(te_tokens), 128):
            chunk = te_tokens[i:i + 128]
            ids, cas, _ = batchify(chunk)
            out = model(ids, cas).argmax(-1).cpu().numpy()
            preds += [[LABELS[k] for k in out[j, :len(s)]] for j, s in enumerate(chunk)]
    return preds, losses

arrs = []
for sd in CFG["ner_seeds"]:
    pr, l = run_lstm_tagger(sd)
    arrs.append(entity_counts(te_tags, pr))
HIST["ner_bilstm"] = l
PRED["ner_bilstm"] = arrs
print("BiLSTM F1:", np.mean([entity_f1_stat(a) for a in arrs]))

BiLSTM F1: 0.7604737119737042


In [32]:
from transformers import AutoModelForTokenClassification

NER_MODEL = "distilbert-base-cased"
ner_tok = AutoTokenizer.from_pretrained(NER_MODEL)
l2i = {l: i for i, l in enumerate(LABELS)}

def encode_tokens(tokens, tags=None):
    enc = ner_tok(tokens, is_split_into_words=True, truncation=True, max_length=128)
    items, labs, first_pos = [], [], []
    for i in range(len(tokens)):
        wid = enc.word_ids(i)
        prev, lab, pos = None, [], {}
        for k, w in enumerate(wid):
            if w is None or w == prev:
                lab.append(-100)
            else:
                lab.append(l2i[tags[i][w]] if tags is not None else 0)
                pos[w] = k
            prev = w
        items.append({"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i]})
        labs.append(lab)
        first_pos.append(pos)
    return items, labs, first_pos

ner_tr_items, ner_tr_labs, _ = encode_tokens(tr_tokens, tr_tags)
ner_te_items, _, ner_te_pos = encode_tokens(te_tokens, te_tags)

# веса классов: корень из обратной частоты, нормированный на среднее
cnt = np.bincount([l for seq in ner_tr_labs for l in seq if l != -100], minlength=len(LABELS)).astype(float)
cls_w = 1.0 / np.sqrt(np.maximum(cnt, 1))
cls_w = torch.tensor(cls_w / cls_w.mean(), dtype=torch.float)

def run_ner_distil(variant, seed):
    set_seed(seed)
    model = AutoModelForTokenClassification.from_pretrained(NER_MODEL, num_labels=len(LABELS)).to(DEVICE)
    if variant == "diff_lr":
        groups = [{"params": model.distilbert.parameters(), "lr": 5e-5},
                  {"params": model.classifier.parameters(), "lr": 1e-3}]
    else:
        groups = [{"params": model.parameters(), "lr": 5e-5}]
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    bs, epochs = 32, CFG["ner_epochs"]
    sched = make_scheduler(opt, epochs * math.ceil(len(ner_tr_items) / bs))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    weight = cls_w.to(DEVICE) if variant == "weighted" else None
    losses = []
    for ep in range(epochs):
        model.train()
        order = np.random.permutation(len(ner_tr_items))
        tot = 0
        for i in range(0, len(order), bs):
            idx = order[i:i + bs]
            batch = collate_pad(ner_tok, [ner_tr_items[j] for j in idx], [ner_tr_labs[j] for j in idx])
            batch = to_dev(batch)
            labels = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(**batch).logits
            loss = nn.functional.cross_entropy(logits.float().transpose(1, 2), labels,
                                               weight=weight, ignore_index=-100)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(idx)
        losses.append(tot / len(order))
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(ner_te_items), 128):
            batch = to_dev(collate_pad(ner_tok, ner_te_items[i:i + 128]))
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(**batch).logits.argmax(-1).cpu().numpy()
            for j in range(out.shape[0]):
                pos = ner_te_pos[i + j]
                n_words = len(te_tokens[i + j])
                preds.append([LABELS[out[j, pos[w]]] if w in pos else "O" for w in range(n_words)])
    del model, opt
    torch.cuda.empty_cache()
    return preds, losses

NER_PRED = {}
for variant in ["base", "weighted", "diff_lr"]:
    t0 = time.time()
    arrs = []
    for sd in CFG["ner_seeds"]:
        pr, l = run_ner_distil(variant, sd)
        arrs.append(entity_counts(te_tags, pr))
        if sd == CFG["ner_seeds"][0]:
            NER_PRED[variant] = pr
    PRED[f"ner_{variant}"] = arrs
    HIST[f"ner_{variant}"] = l
    print(variant, "F1", round(np.mean([entity_f1_stat(a) for a in arrs]), 4), f"{time.time()-t0:.0f}s")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_7114/2118384783.py:62: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first valu

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


base F1 0.8673 75s


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

... (5 outputs left out)

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


diff_lr F1 0.8706 75s


In [33]:
ner_methods = [("CRF (признаки вручную)", "ner_crf"), ("BiLSTM", "ner_bilstm"),
               ("DistilBERT (базовый)", "ner_base"),
               ("DistilBERT + веса классов", "ner_weighted"),
               ("DistilBERT + раздельные скорости обучения", "ner_diff_lr")]
for name, key in ner_methods:
    f1 = bootstrap_stat(PRED[key], entity_f1_stat, CFG["n_boot"])
    sd = float(np.std([entity_f1_stat(a) for a in PRED[key]])) if len(PRED[key]) > 1 else 0.0
    add_row("CoNLL-2003", name, dict(f1=f1[0], f1_lo=f1[1], f1_hi=f1[2], f1_std=sd))
for key in ["ner_weighted", "ner_diff_lr"]:
    print(key, "минус базовый:", paired_bootstrap(PRED[key], PRED["ner_base"], entity_f1_stat, CFG["n_boot"]))

# F1 по типам сущностей для базового DistilBERT (seed 0)
per_type = {}
for t in ["PER", "ORG", "LOC", "MISC"]:
    g = [{e for e in get_entities(s) if e[0] == t} for s in te_tags]
    p = [{e for e in get_entities(s) if e[0] == t} for s in NER_PRED["base"]]
    tp = sum(len(a & b) for a, b in zip(g, p)); np_ = sum(len(b) for b in p); ng = sum(len(a) for a in g)
    pr_, rc_ = tp / max(np_, 1), tp / max(ng, 1)
    per_type[t] = 2 * pr_ * rc_ / max(pr_ + rc_, 1e-9)
print(per_type)
json.dump(per_type, open(f"{OUT}/ner_per_type.json", "w"))

from sklearn.metrics import confusion_matrix
yt = [l for s in te_tags for l in s]
yp = [l for s in NER_PRED["base"] for l in s]
types = [l for l in LABELS]
cm = confusion_matrix(yt, yp, labels=types)
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(types))); ax.set_xticklabels(types, rotation=45, ha="right")
ax.set_yticks(range(len(types))); ax.set_yticklabels(types)
ax.set_xlabel("Предсказанная метка"); ax.set_ylabel("Истинная метка")
fig.colorbar(im, fraction=0.046)
fig.tight_layout(); fig.savefig(f"{OUT}/figures/ner_confusion.png", dpi=200); plt.close(fig)

ner_weighted минус базовый: (-0.009341343327108342, -0.014051517488273208, -0.004774701045549266, 1.0)
ner_diff_lr минус базовый: (0.0032870474617754386, 0.00020465762436102934, 0.006470586088008162, 0.018)
{'PER': 0.9407730673316708, 'ORG': 0.830166270783848, 'LOC': 0.8984816909794582, 'MISC': 0.7058823529411764}


## Задача 3. CNN/DailyMail (суммаризация)

In [34]:
from transformers import AutoModelForSeq2SeqLM

def take(split, n, seed):
    ds = load_dataset("abisee/cnn_dailymail", "3.0.0", split=split, streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=2000)
    return list(itertools.islice(ds, n))

sum_tr = take("train", CFG["sum_train"], 42)
sum_te = take("test", CFG["sum_test"], 42)
tr_art, tr_sum = [x["article"] for x in sum_tr], [x["highlights"] for x in sum_tr]
te_art, te_ref = [x["article"] for x in sum_te], [x["highlights"] for x in sum_te]
print(len(tr_art), len(te_art))

SUM_MODEL = "t5-small"
sum_tok = AutoTokenizer.from_pretrained(SUM_MODEL)

def rouge_row(key, name, preds):
    r = rouge_per_example(preds, te_ref)
    PRED[key] = r
    out = {}
    for j, m in enumerate(["rouge1", "rouge2", "rougeL"]):
        p, lo, hi = bootstrap_stat([r[:, j]], lambda a: float(a.mean()), CFG["n_boot"])
        out[m], out[m + "_lo"], out[m + "_hi"] = p, lo, hi
    out["words"] = float(np.mean([len(x.split()) for x in preds]))
    add_row("CNN/DailyMail", name, out)
    print(name, {k: round(v, 4) for k, v in out.items() if k in ("rouge1", "rouge2", "rougeL", "words")})

rouge_row("sum_lead3", "Lead-3", [lead3(a) for a in te_art])

2000 300
Lead-3 {'rouge1': 0.3368, 'rouge2': 0.1401, 'rougeL': 0.2244, 'words': 71.11}


In [35]:
def generate(model, articles, num_beams=4, no_repeat=3, max_new=96, bs=16):
    model.eval()
    outs = []
    for i in range(0, len(articles), bs):
        chunk = ["summarize: " + a for a in articles[i:i + bs]]
        enc = sum_tok(chunk, return_tensors="pt", truncation=True, max_length=512, padding=True).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                                 num_beams=num_beams, no_repeat_ngram_size=no_repeat, max_new_tokens=max_new)
        outs += sum_tok.batch_decode(ids, skip_special_tokens=True)
    return outs

def finetune_t5(use_lora, seed):
    set_seed(seed)
    model = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL)
    lr_ = 3e-4
    if use_lora:
        cfg = LoraConfig(task_type="SEQ_2_SEQ_LM", r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q", "v"])
        model = get_peft_model(model, cfg)
        lr_ = 1e-3
    model.to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr_, weight_decay=0.01)
    bs, epochs = 8, CFG["sum_epochs"]
    sched = make_scheduler(opt, epochs * math.ceil(len(tr_art) / bs))
    X = sum_tok(["summarize: " + a for a in tr_art], truncation=True, max_length=512)
    Y = sum_tok(text_target=tr_sum, truncation=True, max_length=96)
    items = [{"input_ids": X["input_ids"][i], "attention_mask": X["attention_mask"][i]} for i in range(len(tr_art))]
    losses = []
    for ep in range(epochs):
        model.train()
        order = np.random.permutation(len(items))
        tot = 0
        for i in range(0, len(order), bs):
            idx = order[i:i + bs]
            batch = to_dev(collate_pad(sum_tok, [items[j] for j in idx], [Y["input_ids"][j] for j in idx]))
            loss = model(**batch).loss
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step()
            tot += loss.item() * len(idx)
        losses.append(tot / len(order))
    return model, losses, sum(p.numel() for p in params)

base = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL).to(DEVICE)
rouge_row("sum_zero", "T5-small без дообучения", generate(base, te_art))
del base
torch.cuda.empty_cache()

SUM_MODELS = {}
for name, use_lora, key in [("T5-small, полное дообучение", False, "sum_full"), ("T5-small + LoRA", True, "sum_lora")]:
    t0 = time.time()
    model, l, ntr = finetune_t5(use_lora, CFG["sum_seeds"][0])
    HIST[key] = l
    rouge_row(key, name, generate(model, te_art))
    ROWS[-1]["trainable"] = ntr
    print(f"{name}: {time.time()-t0:.0f}s, обучаемых параметров {ntr}")
    if not use_lora:
        SUM_MODELS["full"] = model
    else:
        del model
        torch.cuda.empty_cache()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5-small без дообучения {'rouge1': 0.35, 'rouge2': 0.1455, 'rougeL': 0.2472, 'words': 44.3267}


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5-small, полное дообучение {'rouge1': 0.3711, 'rouge2': 0.1587, 'rougeL': 0.2572, 'words': 50.6267}
T5-small, полное дообучение: 428s, обучаемых параметров 60506624


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5-small + LoRA {'rouge1': 0.3705, 'rouge2': 0.1619, 'rougeL': 0.2573, 'words': 50.66}
T5-small + LoRA: 354s, обучаемых параметров 589824


In [ ]:
# параметры декодирования для дообученной модели
dec = [("жадный поиск", dict(num_beams=1, no_repeat=0)),
       ("beam search, 4 луча", dict(num_beams=4, no_repeat=0)),
       ("beam search + запрет повторов 3-грамм", dict(num_beams=4, no_repeat=3))]
for name, kw in dec:
    preds = generate(SUM_MODELS["full"], te_art, **kw)
    rouge_row("sum_dec_" + name, "Декодирование: " + name, preds)

d = paired_bootstrap([PRED["sum_full"][:, 0]], [PRED["sum_zero"][:, 0]], lambda a: float(a.mean()), CFG["n_boot"])
print("ROUGE-1, дообучение минус без дообучения:", d)

## Итоги и графики

In [ ]:
df = pd.DataFrame(ROWS)
df.to_csv(f"{OUT}/results.csv", index=False)
json.dump({k: v for k, v in HIST.items()}, open(f"{OUT}/history.json", "w"))
print(df.round(4).to_string())

# рисунок: сравнение методов на SST-2
s = df[df.task == "SST-2"]
fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(s))
ax.barh(y, s.acc * 100, xerr=[(s.acc - s.acc_lo) * 100, (s.acc_hi - s.acc) * 100], color="#4C72B0", capsize=3)
ax.set_yticks(y); ax.set_yticklabels(s.method); ax.invert_yaxis()
ax.set_xlim(max(0, s.acc.min() * 100 - 5), 100); ax.set_xlabel("Accuracy, %")
fig.tight_layout(); fig.savefig(f"{OUT}/figures/sst2_methods.png", dpi=200); plt.close(fig)

# рисунок: кривые обучения DistilBERT
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for key, label in [("sst_full", "полное"), ("sst_freeze", "заморозка"), ("sst_lora", "LoRA")]:
    axes[0].plot(range(1, len(HIST[key]) + 1), HIST[key], marker="o", label=label)
axes[0].set_title("SST-2"); axes[0].set_xlabel("Эпоха"); axes[0].set_ylabel("Потери на обучении"); axes[0].legend()
for key, label in [("ner_base", "базовый"), ("ner_weighted", "веса классов"), ("ner_diff_lr", "раздельные lr")]:
    axes[1].plot(range(1, len(HIST[key]) + 1), HIST[key], marker="o", label=label)
axes[1].set_title("CoNLL-2003"); axes[1].set_xlabel("Эпоха"); axes[1].legend()
fig.tight_layout(); fig.savefig(f"{OUT}/figures/training_curves.png", dpi=200); plt.close(fig)

# рисунок: ROUGE
m = df[(df.task == "CNN/DailyMail") & (~df.method.str.startswith("Декодирование"))]
fig, ax = plt.subplots(figsize=(8, 4))
w = 0.25
for j, col in enumerate(["rouge1", "rouge2", "rougeL"]):
    ax.bar(np.arange(len(m)) + (j - 1) * w, m[col] * 100, w, label=col.upper().replace("ROUGE", "ROUGE-"))
ax.set_xticks(range(len(m))); ax.set_xticklabels(m.method, rotation=15, ha="right")
ax.set_ylabel("F-мера, %"); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/figures/rouge.png", dpi=200); plt.close(fig)

# примеры суммаризации для приложения
ex = {"article": te_art[0][:700], "reference": te_ref[0]}
ex["lead3"] = lead3(te_art[0])
ex["t5_finetuned"] = generate(SUM_MODELS["full"], te_art[:1])[0]
json.dump(ex, open(f"{OUT}/example.json", "w"), ensure_ascii=False, indent=2)

In [38]:
import shutil
shutil.make_archive("results", "zip", OUT)
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("архив results.zip создан в рабочей папке")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>